In [ ]:
import pandas as pd
import glob
import os
import seaborn as sns
import matplotlib.pyplot as plt

data_dir = 'tennis_atp-master'
all_files = glob.glob(os.path.join(data_dir, "atp_matches_*.csv"))

df_list = [pd.read_csv(f) for f in all_files]
df = pd.concat(df_list, ignore_index=True)
df['tourney_date'] = pd.to_datetime(df['tourney_date'], format='%Y%m%d', errors='coerce')

#Correlation Heatmap
numeric_cols = ['minutes', 'w_ace', 'w_df', 'w_svpt', 'w_1stIn', 'w_1stWon', 'l_ace', 'l_df']
corr = df[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation Matrix: Match Statistics")
plt.show()

# 5. Rolling Average Plot: Averages of First Serve Points Won over Time
df = df.sort_values('tourney_date')
df['w_1stWon_pct'] = (df['w_1stWon'] / df['w_1stIn']) * 100
rolling_avg = df.set_index('tourney_date')['w_1stWon_pct'].rolling(window=500).mean()

plt.figure(figsize=(12, 6))
rolling_avg.plot()
plt.title("500-Match Rolling Average: Winner's 1st Serve Points Won %")
plt.ylabel("Percentage")
plt.show()

A clear shift away from zero. If the peak is significantly positive, "Ace Difference" is a major driver of winning.

In [ ]:
data_dir = 'tennis_atp-master'
all_files = glob.glob(os.path.join(data_dir, "atp_matches_*.csv"))

# Read and concatenate all match files
df_list = [pd.read_csv(f) for f in all_files]
df = pd.concat(df_list, ignore_index=True)

# Calculate Ace Difference
# Positive values indicate the winner hit more aces than the loser
df['ace_diff'] = df['w_ace'] - df['l_ace']

# Drop NaN values for accurate plotting
ace_data = df['ace_diff'].dropna()

# Plot the distribution
plt.figure(figsize=(10, 6))
sns.kdeplot(ace_data, fill=True, color="blue", label="Ace Difference")
plt.axvline(0, color='red', linestyle='--', label="Zero Difference")
plt.title("Distribution of Ace Difference (Winner - Loser)")
plt.xlabel("Ace Advantage ($w\_ace - l\_ace$)")
plt.ylabel("Density")
plt.legend()
plt.grid(alpha=0.3)

# Display statistics to verify the "shift"
mean_diff = ace_data.mean()
print(f"Mean Ace Difference: {mean_diff:.2f}")

plt.show()

Tennis skill is not universal. This box plot reveals which surface-specific features are most volatile.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure the surface column is cleaned to handle potential whitespace
df['surface'] = df['surface'].str.strip()

# Filter for major surfaces
surfaces = df[df['surface'].isin(['Hard', 'Clay', 'Grass'])]

# Create the boxplot
plt.figure(figsize=(12, 6))
sns.boxplot(x='surface', y='w_1stWon', data=surfaces)
plt.title("Winner's 1st Serve Points Won by Surface")
plt.xlabel("Surface")
plt.ylabel("Winner's 1st Serve Points Won")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure rankings are numeric (drop matches where rank is missing)
df['winner_rank'] = pd.to_numeric(df['winner_rank'], errors='coerce')
df['loser_rank'] = pd.to_numeric(df['loser_rank'], errors='coerce')
plot_data = df.dropna(subset=['winner_rank', 'loser_rank', 'surface']).copy()

# Calculate Rank Difference (L - W)
# A positive value means the winner was ranked higher (e.g., #5 beats #50 -> 50 - 5 = 45)
# A negative value means the lower ranked player won (an upset)
plot_data['rank_diff'] = plot_data['loser_rank'] - plot_data['winner_rank']

# Filter for major surfaces
plot_data = plot_data[plot_data['surface'].isin(['Hard', 'Clay', 'Grass'])]

# Plot
plt.figure(figsize=(12, 6))
sns.boxplot(x='surface', y='rank_diff', data=plot_data, palette='viridis', fliersize=3)
plt.axhline(0, color='red', linestyle='--', label='Equal Ranking')
plt.title("Distribution of Ranking Difference (Loser Rank - Winner Rank) by Surface")
plt.ylabel("Ranking Gap (Winner's Relative Strength)")
plt.xlabel("Surface")
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

Hard Courts: Exhibit the largest spread. This suggests that "upsets" (where a significantly lower-rated player wins) are more common or possible on Hard courts compared to others.

Clay/Grass: The distribution is tighter. This suggests that on these surfaces, the Elo rating is a slightly more reliable indicator of the outcome.

The Outliers: Those circles far above and below the boxes are "upsets"—matches where an extreme underdog beat an extreme favorite.
